# Small Model Comparison - Google Colab Setup

This notebook sets up the small model comparison project in Google Colab.

## 1. Clone Repository and Install Dependencies

In [ ]:
# Clone the repository (replace with your actual repo URL)
!git clone https://github.com/yourusername/small-model-comparison.git
%cd small-model-comparison

In [ ]:
# Install requirements
!pip install -q -r requirements.txt
!pip install -q accelerate bitsandbytes  # For model optimization

## 2. Check Hardware and Setup

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Check available RAM and disk space
import psutil
import shutil

ram_gb = psutil.virtual_memory().total / (1024**3)
disk_gb = shutil.disk_usage('/').free / (1024**3)

print(f"Available RAM: {ram_gb:.1f} GB")
print(f"Available Disk: {disk_gb:.1f} GB")

## 3. Hugging Face Setup

In [ ]:
# Setup Hugging Face authentication (for gated models like LLaMA)
from huggingface_hub import login
from getpass import getpass

# Uncomment and run if you need to login
# hf_token = getpass("Enter your Hugging Face token: ")
# login(token=hf_token)

## 4. Download Models (Optimized for Colab)

In [ ]:
# Colab-optimized model download function
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

def load_model_colab(model_name, use_cache=True):
    """Load model optimized for Colab environment."""
    print(f"Loading {model_name}...")
    
    # Load with optimization for limited memory
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,  # Use FP16 to save memory
        device_map="auto",          # Automatic device placement
        trust_remote_code=True,
        use_cache=use_cache
    )
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print(f"✅ {model_name} loaded successfully")
    return model, tokenizer

## 5. Model Comparison Setup

In [ ]:
# Define models for comparison (Colab-friendly list)
MODELS = {
    "llama-3.2-1b": "meta-llama/Llama-3.2-1B-Instruct",
    "llama-3.2-3b": "meta-llama/Llama-3.2-3B-Instruct", 
    "phi-3-mini": "microsoft/Phi-3-mini-4k-instruct",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # Add others as needed
}

print("Models configured for comparison:")
for name, hf_name in MODELS.items():
    print(f"  {name}: {hf_name}")

## 6. Memory Management for Colab

In [ ]:
# Memory cleanup function
def cleanup_memory():
    """Clean up GPU and CPU memory."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Memory cleaned up")

# Function to check memory usage
def check_gpu_memory():
    """Check current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        cached = torch.cuda.memory_reserved(0) / 1e9
        print(f"GPU Memory - Allocated: {allocated:.1f}GB, Cached: {cached:.1f}GB")
    else:
        print("No GPU available")

## 7. Ready to Start!

In [ ]:
print("🚀 Setup complete! Ready for model comparison.")
print("\nNext steps:")
print("1. Load your first model: model, tokenizer = load_model_colab('TinyLlama/TinyLlama-1.1B-Chat-v1.0')")
print("2. Run benchmarks on loaded models")
print("3. Compare results")
print("\n⚠️  Colab Tips:")
print("- Use cleanup_memory() between model loads")
print("- Monitor GPU memory with check_gpu_memory()")
print("- Save results to Google Drive for persistence")